# 2s-AGCN Violence Detection Training (RWF-2000)

In this notebook, we train a 2s-AGCN (Two-Stream Adaptive Graph Convolutional Network) model for violence detection using skeleton pose data extracted from the RWF-2000 dataset.

## What is done
- Load pre-extracted pose keypoints from `.pkl` annotation file
- Configure MMAction2's 2s-AGCN model for binary classification (Fight / NonFight)
- Train the GCN model on skeleton sequences using joint stream features
- Evaluate on validation set after each epoch and save best checkpoint

## Model
- **Backbone**: AAGCN (with `gcn_attention=False` → degrades to standard AGCN)
- **Graph layout**: COCO 17-keypoint skeleton
- **Classifier**: GCNHead with 2 output classes
- **Optimizer**: SGD with CosineAnnealingLR scheduler

## Input
Pre-processed pose annotation file:
`data/processed/pose/pkl/rwf_pose.pkl`

Each annotation contains:
- `keypoint`: shape `(2, T, 17, 2)` — x, y coordinates
- `keypoint_score`: shape `(2, T, 17)` — confidence scores
- `label`: 0 = NonFight, 1 = Fight

## Output
Best model checkpoint saved to:
`checkpoints/pose/2s-agcn/best_acc_top1_epoch_X.pth`

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


MMAction2 Installation

In [2]:
import torch

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device:", torch.cuda.get_device_name(0))

KeyboardInterrupt: 

In [ ]:
!pip install mmcv==2.1.0 -f https://download.openmmlab.com/mmcv/dist/cu128/torch2.10/index.html

In [ ]:
%cd /content
!git clone https://github.com/open-mmlab/mmaction2.git -q
%cd /content/mmaction2
!pip install -e . -q

In [ ]:
import torch
import numpy as np
print("torch:", torch.__version__)
print("numpy:", np.__version__)
print("cuda:", torch.cuda.is_available())

Restart Kernel

In [ ]:
import os
os.kill(os.getpid(), 9)

In [ ]:
import torch, mmcv, mmengine, mmaction
print("torch:", torch.__version__)
print("mmcv:", mmcv.__version__)
print("mmengine:", mmengine.__version__)
print("mmaction:", mmaction.__version__)

Save mmcv whell to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import glob, shutil
wheels = glob.glob('/root/.cache/pip/wheels/**/*mmcv*.whl', recursive=True)
print("Found wheels:", wheels)
for w in wheels:
    shutil.copy(w, '/content/drive/MyDrive/')
    print("Saved:", w)

in next session

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import glob
wheels = glob.glob('/content/drive/MyDrive/mmcv*.whl')
print(wheels)

['/content/drive/MyDrive/mmcv-2.2.0-cp312-cp312-linux_x86_64.whl', '/content/drive/MyDrive/mmcv-2.1.0-cp312-cp312-linux_x86_64.whl']


In [5]:
!pip install /content/drive/MyDrive/mmcv-2.1.0-cp312-cp312-linux_x86_64.whl

Processing ./drive/MyDrive/mmcv-2.1.0-cp312-cp312-linux_x86_64.whl
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 452.7/452.7 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.2/256.2 kB 16.0 MB/s eta 0:00:00


In [6]:
!pip install mmengine -q
%cd /content
!git clone https://github.com/open-mmlab/mmaction2.git -q
%cd /content/mmaction2
!pip install -e . -q

/content
/content/mmaction2
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 111.0 MB/s eta 0:00:00


In [7]:
!sed -i 's/from .multimodal import \*/# from .multimodal import */' \
  /content/mmaction2/mmaction/models/__init__.py

In [ ]:
import os
os.kill(os.getpid(), 9)

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import torch, mmcv, mmengine, mmaction
print("torch:", torch.__version__)
print("mmcv:", mmcv.__version__)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
torch: 2.10.0+cu128
mmcv: 2.1.0


# 2s-AGCN - Bone Stream

In [3]:
import pickle
import numpy as np

# Verify fixed pkl
pkl_path = '/content/drive/MyDrive/violence-detection/data/processed/pose/pkl/rwf_pose.pkl'
with open(pkl_path, 'rb') as f:
    data = pickle.load(f)

print("Train:", len(data['split']['train']))
print("Val:", len(data['split']['val']))
print("Overlap:", len(set(data['split']['train']) & set(data['split']['val'])))
print("Sample frame_dir:", data['annotations'][0]['frame_dir'])

Train: 1599
Val: 400
Overlap: 0
Sample frame_dir: train/Fight/000000


In [4]:
config_content = """
_base_ = '../../_base_/default_runtime.py'

model = dict(
    type='RecognizerGCN',
    backbone=dict(
        type='AAGCN',
        graph_cfg=dict(layout='coco', mode='spatial'),
        gcn_attention=False),
    cls_head=dict(type='GCNHead', num_classes=2, in_channels=256))

dataset_type = 'PoseDataset'
ann_file = '/content/drive/MyDrive/violence-detection/data/processed/pose/pkl/rwf_pose.pkl'

train_pipeline = [
    dict(type='PreNormalize2D'),
    dict(type='GenSkeFeat', dataset='coco', feats=['b']),
    dict(type='UniformSampleFrames', clip_len=100),
    dict(type='PoseDecode'),
    dict(type='FormatGCNInput', num_person=2),
    dict(type='PackActionInputs')
]
val_pipeline = [
    dict(type='PreNormalize2D'),
    dict(type='GenSkeFeat', dataset='coco', feats=['b']),
    dict(type='UniformSampleFrames', clip_len=100, num_clips=1, test_mode=True),
    dict(type='PoseDecode'),
    dict(type='FormatGCNInput', num_person=2),
    dict(type='PackActionInputs')
]
test_pipeline = [
    dict(type='PreNormalize2D'),
    dict(type='GenSkeFeat', dataset='coco', feats=['b']),
    dict(type='UniformSampleFrames', clip_len=100, num_clips=10, test_mode=True),
    dict(type='PoseDecode'),
    dict(type='FormatGCNInput', num_person=2),
    dict(type='PackActionInputs')
]

train_dataloader = dict(
    batch_size=16,
    num_workers=2,
    persistent_workers=True,
    sampler=dict(type='DefaultSampler', shuffle=True),
    dataset=dict(
        type='RepeatDataset',
        times=5,
        dataset=dict(
            type=dataset_type,
            ann_file=ann_file,
            pipeline=train_pipeline,
            split='train')))
val_dataloader = dict(
    batch_size=16,
    num_workers=2,
    persistent_workers=True,
    sampler=dict(type='DefaultSampler', shuffle=False),
    dataset=dict(
        type=dataset_type,
        ann_file=ann_file,
        pipeline=val_pipeline,
        split='val',
        test_mode=True))
test_dataloader = dict(
    batch_size=1,
    num_workers=2,
    persistent_workers=True,
    sampler=dict(type='DefaultSampler', shuffle=False),
    dataset=dict(
        type=dataset_type,
        ann_file=ann_file,
        pipeline=test_pipeline,
        split='val',
        test_mode=True))

val_evaluator = [dict(type='AccMetric')]
test_evaluator = val_evaluator

train_cfg = dict(
    type='EpochBasedTrainLoop', max_epochs=16, val_begin=1, val_interval=1)
val_cfg = dict(type='ValLoop')
test_cfg = dict(type='TestLoop')

param_scheduler = [
    dict(
        type='CosineAnnealingLR',
        eta_min=0,
        T_max=16,
        by_epoch=True,
        convert_to_iter_based=True)
]

optim_wrapper = dict(
    optimizer=dict(
        type='SGD', lr=0.1, momentum=0.9, weight_decay=0.0005, nesterov=True))

default_hooks = dict(checkpoint=dict(interval=1), logger=dict(interval=50))
auto_scale_lr = dict(enable=False, base_batch_size=128)
"""

with open('/content/mmaction2/configs/skeleton/2s-agcn/2s-agcn_rwf2000_bone.py', 'w') as f:
    f.write(config_content)

print("Bone-stream config saved!")

Bone-stream config saved!


Train

In [ ]:
%cd /content/mmaction2
!python tools/train.py configs/skeleton/2s-agcn/2s-agcn_rwf2000_bone.py

/content/mmaction2
05/16 13:11:40 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
    CUDA available: True
    MUSA available: False
    numpy_random_seed: 439119147
    GPU 0: Tesla T4
    CUDA_HOME: /usr/local/cuda
    NVCC: Cuda compilation tools, release 12.8, V12.8.93
    GCC: x86_64-linux-gnu-gcc (Ubuntu 11.4.0-1ubuntu1~22.04.3) 11.4.0
    PyTorch: 2.10.0+cu128
    PyTorch compiling details: PyTorch built with:
  - GCC 13.3
  - C++ Version: 201703
  - Intel(R) oneAPI Math Kernel Library Version 2024.2-Product Build 20240605 for Intel(R) 64 architecture applications
  - Intel(R) MKL-DNN v3.7.1 (Git Hash 8d263e693366ef8db40acc569cc7d8edf644556d)
  - OpenMP 201511 (a.k.a. OpenMP 4.5)
  - LAPACK is enabled (usually provided by MKL)
  - NNPACK is enabled
  - CPU capability usage: AVX512
  - CUDA Runtime 12.8
  - NVCC architecture flags: -gencode;

In [ ]:
import re, json, glob, shutil

log_files = glob.glob(
    '/content/mmaction2/work_dirs/2s-agcn_rwf2000_bone/**/*.log',
    recursive=True
)
print("Found logs:", log_files)

bone_val_accs = []
for lf in log_files:
    with open(lf) as f:
        for line in f:
            m = re.search(r'Epoch\(val\).*acc/top1:\s*([\d.]+)', line)
            if m:
                bone_val_accs.append(float(m.group(1)))

print("Bone val accs:", bone_val_accs)

with open('/content/drive/MyDrive/violence-detection/results/bone_val_accs.json', 'w') as f:
    json.dump({'val_acc': bone_val_accs}, f)

for lf in log_files:
    shutil.copy(lf, '/content/drive/MyDrive/violence-detection/results/bone_training.log')

print("Saved!")

Save the best model

In [ ]:
import shutil, glob, os

dst = '/content/drive/MyDrive/violence-detection/checkpoints/pose/2s-agcn_bone/'
os.makedirs(dst, exist_ok=True)

best = glob.glob('/content/mmaction2/work_dirs/2s-agcn_rwf2000_bone/best_acc_top1_epoch_*.pth')

for f in best:
    shutil.copy(f, dst)
    print("Saved:", f)